# Uncertainty-Aware WR Separation Leaderboard

The baseline notebook showed that the wide receiver model is the cleanest first surface for this project. This notebook focuses on the player-ranking question:

> Which wide receivers create or preserve more separation than expected, and how confident are we that the effect is not just small-sample noise?

The route-level score is:

```text
SOE_route = delta_sep_actual - delta_sep_expected
```

The player summary reports the mean route-level SOE and an approximate 95% confidence interval:

```text
lower_95_soe = mean_soe - 1.96 * standard_error
```

The conservative leaderboard sorts by `lower_95_soe`.

In [1]:
from pathlib import Path
import csv
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from separation_over_expected.reports import read_csv_rows

SUMMARY_PATH = PROJECT_ROOT / "data" / "processed" / "position_baselines" / "receiver_baseline_summary_wr.csv"
METRICS_PATH = PROJECT_ROOT / "data" / "processed" / "position_baselines" / "baseline_metrics_wr.csv"

PROJECT_ROOT

PosixPath('/Users/moggo/Projects/NFL/Separation_Over_Expected')

## Model Check

The WR-specific model is fit only on WR route observations. That makes the comparison group cleaner than the pooled model because outside receivers, slot receivers, and WR motion/alignment patterns are no longer mixed with TE and RB route jobs.

In [2]:
metrics = read_csv_rows(METRICS_PATH)
[row for row in metrics if row["split"] == "test"]

[{'model': 'global_mean', 'split': 'test', 'rows': '2474', 'mae': '2.012', 'rmse': '2.626', 'r2': '-0.004', 'bias': '-0.163', 'mean_actual': '-2.480', 'mean_predicted': '-2.317', 'n': '2474'}, {'model': 'smoothed_group_mean', 'split': 'test', 'rows': '2474', 'mae': '2.003', 'rmse': '2.619', 'r2': '0.002', 'bias': '-0.167', 'mean_actual': '-2.480', 'mean_predicted': '-2.313', 'n': '2474'}, {'model': 'ridge_context', 'split': 'test', 'rows': '2474', 'mae': '1.393', 'rmse': '1.848', 'r2': '0.503', 'bias': '-0.050', 'mean_actual': '-2.480', 'mean_predicted': '-2.430', 'n': '2474'}]

The ridge context model is the first useful expected-separation model for WRs. It explains about half of holdout variation in snap-to-release separation change, while the group-mean baseline barely improves over the global mean.

In [3]:
for row in metrics:
    if row["split"] == "test":
        print(f"{row['model']:>20}  R2={row['r2']}  RMSE={row['rmse']}  MAE={row['mae']}")

         global_mean  R2=-0.004  RMSE=2.626  MAE=2.012
 smoothed_group_mean  R2=0.002  RMSE=2.619  MAE=2.003
       ridge_context  R2=0.503  RMSE=1.848  MAE=1.393


## Conservative Leaderboard

Sorting by `lower_95_soe` gives a conservative view: players need both a positive average and enough route volume to separate from zero. This is the table I would show first in a portfolio writeup.

In [4]:
wr_summary = read_csv_rows(SUMMARY_PATH)
wr_summary[:15]

[{'nflId': '53478', 'displayName': 'Rondale Moore', 'officialPosition': 'WR', 'routes': '97', 'mean_soe': '1.932', 'median_soe': '1.134', 'std_soe': '3.792', 'se_soe': '0.385', 'lower_95_soe': '1.177', 'upper_95_soe': '2.687', 'total_soe': '187.421'}, {'nflId': '52442', 'displayName': 'Michael Pittman', 'officialPosition': 'WR', 'routes': '218', 'mean_soe': '0.409', 'median_soe': '0.103', 'std_soe': '1.948', 'se_soe': '0.132', 'lower_95_soe': '0.150', 'upper_95_soe': '0.667', 'total_soe': '89.134'}, {'nflId': '47839', 'displayName': 'Mecole Hardman', 'officialPosition': 'WR', 'routes': '201', 'mean_soe': '0.468', 'median_soe': '-0.198', 'std_soe': '2.342', 'se_soe': '0.165', 'lower_95_soe': '0.145', 'upper_95_soe': '0.792', 'total_soe': '94.159'}, {'nflId': '52622', 'displayName': 'Freddie Swain', 'officialPosition': 'WR', 'routes': '147', 'mean_soe': '0.521', 'median_soe': '-0.185', 'std_soe': '2.574', 'se_soe': '0.212', 'lower_95_soe': '0.105', 'upper_95_soe': '0.937', 'total_soe': '

## Raw Mean Leaderboard

The raw mean leaderboard answers a different question: who had the highest average SOE, regardless of uncertainty? Comparing this with the conservative leaderboard shows which players are sample-size-sensitive.

In [5]:
raw_mean_leaders = sorted(wr_summary, key=lambda row: float(row["mean_soe"]), reverse=True)
raw_mean_leaders[:15]

[{'nflId': '53478', 'displayName': 'Rondale Moore', 'officialPosition': 'WR', 'routes': '97', 'mean_soe': '1.932', 'median_soe': '1.134', 'std_soe': '3.792', 'se_soe': '0.385', 'lower_95_soe': '1.177', 'upper_95_soe': '2.687', 'total_soe': '187.421'}, {'nflId': '53658', 'displayName': 'Mike Strachan', 'officialPosition': 'WR', 'routes': '30', 'mean_soe': '0.753', 'median_soe': '0.681', 'std_soe': '2.422', 'se_soe': '0.442', 'lower_95_soe': '-0.114', 'upper_95_soe': '1.620', 'total_soe': '22.585'}, {'nflId': '53456', 'displayName': 'Rashod Bateman', 'officialPosition': 'WR', 'routes': '36', 'mean_soe': '0.536', 'median_soe': '0.127', 'std_soe': '2.191', 'se_soe': '0.365', 'lower_95_soe': '-0.180', 'upper_95_soe': '1.252', 'total_soe': '19.294'}, {'nflId': '52622', 'displayName': 'Freddie Swain', 'officialPosition': 'WR', 'routes': '147', 'mean_soe': '0.521', 'median_soe': '-0.185', 'std_soe': '2.574', 'se_soe': '0.212', 'lower_95_soe': '0.105', 'upper_95_soe': '0.937', 'total_soe': '76.

## Players Most Affected By Uncertainty

A simple way to see the effect of uncertainty is to compare each player's raw-mean rank with his conservative rank.

In [6]:
def add_rank(rows, key):
    ranked = sorted(rows, key=lambda row: float(row[key]), reverse=True)
    return {row["nflId"]: rank for rank, row in enumerate(ranked, start=1)}

mean_rank = add_rank(wr_summary, "mean_soe")
lower_rank = add_rank(wr_summary, "lower_95_soe")

rank_changes = []
for row in wr_summary:
    rank_changes.append({
        "displayName": row["displayName"],
        "routes": row["routes"],
        "mean_soe": row["mean_soe"],
        "lower_95_soe": row["lower_95_soe"],
        "mean_rank": mean_rank[row["nflId"]],
        "conservative_rank": lower_rank[row["nflId"]],
        "rank_drop": lower_rank[row["nflId"]] - mean_rank[row["nflId"]],
    })

sorted(rank_changes, key=lambda row: row["rank_drop"], reverse=True)[:15]

[{'displayName': 'Trent Sherfield', 'routes': '39', 'mean_soe': '0.366', 'lower_95_soe': '-0.604', 'mean_rank': 10, 'conservative_rank': 130, 'rank_drop': 120}, {'displayName': 'T.Y. Hilton', 'routes': '34', 'mean_soe': '0.144', 'lower_95_soe': '-0.676', 'mean_rank': 37, 'conservative_rank': 142, 'rank_drop': 105}, {'displayName': 'Dede Westbrook', 'routes': '40', 'mean_soe': '0.048', 'lower_95_soe': '-0.681', 'mean_rank': 60, 'conservative_rank': 144, 'rank_drop': 84}, {'displayName': 'Marquez Valdes-Scantling', 'routes': '58', 'mean_soe': '0.073', 'lower_95_soe': '-0.570', 'mean_rank': 51, 'conservative_rank': 124, 'rank_drop': 73}, {'displayName': 'Anthony Miller', 'routes': '37', 'mean_soe': '0.257', 'lower_95_soe': '-0.383', 'mean_rank': 21, 'conservative_rank': 85, 'rank_drop': 64}, {'displayName': 'Equanimeous St. Brown', 'routes': '32', 'mean_soe': '0.094', 'lower_95_soe': '-0.487', 'mean_rank': 46, 'conservative_rank': 108, 'rank_drop': 62}, {'displayName': 'Noah Brown', 'rout

## Takeaway

The uncertainty-aware view changes the story in the right direction. Small-sample players can still appear if their signal is strong, but larger-sample receivers move up when their positive SOE is more stable. For the next iteration, the natural validation step is split-half stability: do WRs who rate well in weeks 1-4 continue to rate well in weeks 5-8?